# FeS wall inventory → elemental sulfur → warm compressor shaft

Screening example for 1 MSm³/day natural gas containing 20 ppm H₂S, mixed with 10 kg/h nitrogen containing 2 mol% oxygen. The calculation separates the historical FeS wall route from homogeneous H₂S oxidation and carries generated S⁰ forward as NeqSim `S8`. It also includes an adjustable entrained-condensate rate and evaluates evaporation/solid precipitation at a warm shaft.

> The FeS oxidation rate and S⁰ selectivity require plant calibration. Defaults in the Java model are zero; the values below are explicit sensitivity assumptions, not universal kinetics.

In [ ]:
import os
import sys
from pathlib import Path

def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")

PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True)
ns = neqsim_classes(ns)
NEQSIM_MODE = "devtools"

In [ ]:
import math
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

JClass = ns.JClass
SystemSrkEos = JClass("neqsim.thermo.system.SystemSrkEos")
Stream = JClass("neqsim.process.equipment.stream.Stream")
WallInventory = JClass("neqsim.process.equipment.reactor.IronSulfideWallInventory")
WallSource = JClass("neqsim.process.equipment.reactor.IronSulfideOxidationSource")
FeSPhase = JClass("neqsim.process.equipment.reactor.IronSulfideWallInventory$FeSPhase")
SolidFlashDepositSource = JClass("neqsim.process.equipment.compressor.SolidFlashDepositSource")
DepositMechanism = JClass("neqsim.process.equipment.compressor.DepositMechanism")
Compressor = JClass("neqsim.process.equipment.compressor.Compressor")
ThermalModel = JClass("neqsim.process.equipment.compressor.CompressorThermalModel")
ThermalNode = JClass("neqsim.process.equipment.compressor.CompressorThermalNode")
NodeType = JClass("neqsim.process.equipment.compressor.CompressorThermalNode$NodeType")

## Feed and stoichiometric ceilings

The standard-volume conversion below uses 22.414 Sm³/kmol (0 °C basis) only to establish the screening molar flow. Change this if the plant MSm³ definition uses another reference condition.

In [ ]:
gas_MSm3_day = 1.0
h2s_ppm = 20.0
nitrogen_package_kg_h = 10.0
oxygen_mol_fraction_in_package = 0.02
entrained_condensate_kg_h = 20.0  # measured carry-over should replace this assumption

process_kmol_h = gas_MSm3_day * 1e6 / 22.414 / 24.0
package_mw = (1-oxygen_mol_fraction_in_package)*28.0134 + oxygen_mol_fraction_in_package*31.998
package_kmol_h = nitrogen_package_kg_h / package_mw
oxygen_kmol_h = package_kmol_h * oxygen_mol_fraction_in_package
oxygen_kg_h = oxygen_kmol_h * 31.998
h2s_kmol_h = process_kmol_h * h2s_ppm * 1e-6
h2s_kg_h = h2s_kmol_h * 34.081
wall_route_s0_ceiling_kg_h = oxygen_kmol_h / 0.75 * 32.065
homogeneous_route_s0_ceiling_kg_h = min(h2s_kmol_h, 2*oxygen_kmol_h) * 32.065

pd.Series({
    "process gas (kmol/h)": process_kmol_h,
    "H2S in gas (kg/h)": h2s_kg_h,
    "O2 in nitrogen package (kg/h)": oxygen_kg_h,
    "FeS-wall S0 ceiling at 0.75 mol O2/mol FeS (kg/h)": wall_route_s0_ceiling_kg_h,
    "homogeneous H2S-oxidation S0 ceiling (kg/h)": homogeneous_route_s0_ceiling_kg_h,
})

In [ ]:
temperature_C, pressure_bara = 5.0, 40.0
methane_kmol_h = process_kmol_h - h2s_kmol_h
nitrogen_kmol_h = package_kmol_h * (1-oxygen_mol_fraction_in_package)
heptane_kmol_h = entrained_condensate_kg_h / 100.205
total_kmol_h = methane_kmol_h + h2s_kmol_h + nitrogen_kmol_h + oxygen_kmol_h + heptane_kmol_h

fluid = SystemSrkEos(273.15 + temperature_C, pressure_bara)
for name, amount in [("methane", methane_kmol_h), ("H2S", h2s_kmol_h),
                     ("nitrogen", nitrogen_kmol_h), ("oxygen", oxygen_kmol_h),
                     ("n-heptane", heptane_kmol_h), ("S8", 1e-12),
                     ("hydrogen", 1e-12), ("CO2", 1e-12), ("water", 1e-12)]:
    fluid.addComponent(name, amount)
fluid.setMixingRule(2)
fluid.setMultiPhaseCheck(True)
feed = Stream("sour gas + oxygen-containing nitrogen + condensate", fluid)
feed.setFlowRate(total_kmol_h, "kmol/hr")
feed.run()

wall = WallInventory(250.0, 80.0, 20.0)
wall.setPipeSurfaceAreaM2(1200.0)
wall.setPorosityFraction(0.35)
wall.setWettedFraction(0.20)
wall.setSurfaceAreaMultiplier(2.0)
wall.setFeSPhase(FeSPhase.MIXED)

source = WallSource("historical FeS wall source", feed, wall)
source.setFeSOxidationFractionPerHour(0.02)
source.setOxygenTransferFraction(1.0)
source.setElementalSulfurYieldFraction(0.50)
source.setElementalSulfurYieldBounds(0.10, 1.00)
source.setKineticUncertaintyFactor(3.0)
source.run()

pd.Series({
    "FeS inventory (kg)": wall.getFeSMassKg(),
    "maximum S held in FeS inventory (kg)": wall.getMaximumElementalSulfurMassKg(),
    "O2 consumed (kg/h)": source.getOxygenConsumedKgPerHour(),
    "S0 low (kg/h)": source.getElementalSulfurLowRateKgPerHour(),
    "S0 base (kg/h)": source.getElementalSulfurRateKgPerHour(),
    "S0 high (kg/h)": source.getElementalSulfurHighRateKgPerHour(),
    "reaction heat (kW)": source.getReactionHeatKW(),
    "limiting factor": str(source.getOxidationLimitingFactor()),
})

In [ ]:
rates = [source.getElementalSulfurLowRateKgPerHour(), source.getElementalSulfurRateKgPerHour(),
         source.getElementalSulfurHighRateKgPerHour()]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["low", "base", "high"], rates, color=["#7aa6c2", "#e3a64b", "#b95555"])
ax.axhline(wall_route_s0_ceiling_kg_h, color="black", ls="--", label="O2 stoichiometric ceiling")
ax.set_ylabel("Elemental sulfur source (kg/h)")
ax.set_title("Uncalibrated FeS-wall source range")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()

## Entrained condensate at a warm inlet shaft

The same S8-bearing outlet is flashed over local temperature while pressure is held at suction pressure. `getLastLiquidEvaporatedFraction()` compares equilibrium liquid at the bulk inlet with liquid remaining locally. A compressor thermal node can replace the temperature sweep.

In [ ]:
temperatures_C = np.linspace(temperature_C, 120.0, 24)
precipitated, evaporated = [], []
for local_T in temperatures_C:
    dep = SolidFlashDepositSource(source.getOutletStream(), "S8", DepositMechanism.SULFUR_S8, 0.5)
    dep.setEvaluationConditions(float(local_T), pressure_bara)
    precipitated.append(dep.getPrecipitationRate("kg/hr"))
    evaporated.append(dep.getLastLiquidEvaporatedFraction())

compressor = Compressor("recompressor", source.getOutletStream())
thermal = ThermalModel("local measured/estimated temperatures")
thermal.addNode(ThermalNode(ThermalModel.INLET_SHAFT, NodeType.SHAFT, 373.15, 0.0, True))
compressor.setThermalModel(thermal)
shaft_dep = SolidFlashDepositSource(source.getOutletStream(), "S8", DepositMechanism.SULFUR_S8, 0.5)
shaft_dep.setThermalNode(compressor, ThermalModel.INLET_SHAFT)
shaft_precipitation = shaft_dep.getPrecipitationRate("kg/hr")

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(temperatures_C, precipitated, color="#b8860b", lw=2, label="solid S8")
ax1.set_xlabel("Local shaft / surface temperature (°C)")
ax1.set_ylabel("Solid S8 after local flash (kg/h)", color="#8a6508")
ax2 = ax1.twinx()
ax2.plot(temperatures_C, evaporated, color="#236b8e", lw=2, label="condensate evaporated")
ax2.set_ylabel("Fraction of inlet liquid evaporated", color="#236b8e")
ax1.axvline(100.0, color="grey", ls="--", alpha=0.7)
ax1.grid(alpha=0.25)
ax1.set_title("Warm-surface flash: condensate carrier and sulfur solid")
plt.show()

pd.Series({"shaft temperature (°C)": shaft_dep.getLastEvaluationTemperatureC(),
           "liquid evaporated fraction": shaft_dep.getLastLiquidEvaporatedFraction(),
           "S8 precipitated (kg/h)": shaft_precipitation,
           "S8 captured at 50% (kg/h)": shaft_dep.getDepositRate("kg/hr")})

## Interpretation

- The oxygen package supplies about 0.228 kg/h O₂. On the default FeS → Fe₂O₃ + S⁰ branch, this caps S⁰ at about 0.30 kg/h before yield and transfer losses.
- The 20 ppm H₂S stream contains substantially more sulfur than that hourly source, but current gas H₂S alone does not quantify old FeS scale. Scale thickness/mineralogy and operating history determine the available inventory.
- A warm shaft can evaporate the entrained condensate carrier locally. Whether the remaining S8 sticks is represented separately by the capture fraction and should be sensitivity-tested against deposit evidence.
- Compare deposit XRD/SEM-EDS/XPS, oxygen/purge history, separator carry-over and local metal temperatures before selecting a causal route.